In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

from spectral_detection.analysis.pca_pipeline import pipeline

We first load the pipeline that we will be using

In [2]:
pipe = pipeline(L=21, H=32, K=10)

Let us test on a couple of individual data sets first.

In [3]:
X1, ids1 = pipe.load_pt(r"../data/spectral/truthfulQA_Judgelabels_and_eigs_top10.pt")

X1_pca = pipe.fit_transform(X1)

print("Original shape:", X1.shape)
print("Reduced shape:", X1_pca.shape)

Original shape: (817, 6720)
Reduced shape: (817, 335)


In [4]:
X2, ids2 = pipe.load_pt(r"../data/spectral/triviaQA_Judgelabels_and_eigs_top10.pt")

X2_pca = pipe.transform(X2)

print("Original shape:", X2.shape)
print("Reduced shape:", X2_pca.shape)

Original shape: (2000, 6720)
Reduced shape: (2000, 335)


We can now merge all the datasets and do PCA on the whole data.

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [6]:
# -----------------------------
#  List the 4 .pt files
# -----------------------------
pt_paths = [
    # r"../data/spectral/truthfulQA_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/triviaQA_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/mmlu_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/math_Judgelabels_and_eigs_top10.pt"
]

# Optional: dataset names for tracking
dataset_names = ["truthfulqa", "triviaqa", "mmlu", "math"]

# -----------------------------
# Load + featurize all four, then stack
# -----------------------------
X_all = []
ids_all = []
source_all = []

for path, ds_name in zip(pt_paths, dataset_names):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

    X, ids = pipe.load_pt(path, feature_key="eig_top10")  # X shape: [N, L*K] after head-avg + signed-log
    X_all.append(X)
    ids_all.extend(ids)
    source_all.extend([ds_name] * len(ids))

X_all = np.vstack(X_all)  # shape: [N_total, L*K]
print("Combined X shape:", X_all.shape)

Combined X shape: (6000, 6720)


In [7]:
# -----------------------------
# 3) Fit PCA on the combined dataset
# -----------------------------
X_pca = pipe.fit_transform(X_all)
print("PCA output shape:", X_pca.shape)
print("Num PCs kept:", pipe.pca.n_components_)
print("Explained variance retained:", pipe.pca.explained_variance_ratio_.sum())

PCA output shape: (6000, 1378)
Num PCs kept: 1378
Explained variance retained: 0.95005673


In [8]:
# -----------------------------
# 4) Put into a DataFrame for convenience
# -----------------------------
pc_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pc_cols)
df_pca.insert(0, "id", ids_all)
df_pca.insert(1, "dataset", source_all)

df_pca.head()

,id,dataset,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,...,PC1369,PC1370,PC1371,PC1372,PC1373,PC1374,PC1375,PC1376,PC1377,PC1378
0,triviaqa_00000_t0.1_ans00,truthfulqa,-44.182053,11.572464,-0.158500,-8.428147,-12.760209,-0.573039,-3.082951,-2.810615,...,-0.261632,-0.694015,-0.619766,0.251993,-0.325119,0.089933,-0.042068,-0.821198,-0.704396,-0.365106
1,triviaqa_00001_t0.1_ans00,truthfulqa,-47.176624,15.367263,-0.839482,-12.471917,-1.784634,2.181762,13.712010,2.802632,...,-0.008218,0.000905,-0.748834,-0.105519,-0.464503,-0.305665,-0.763447,0.206863,-0.022236,-0.264161
2,triviaqa_00002_t0.1_ans00,truthfulqa,-44.161472,11.508937,10.504965,-1.909707,2.297472,0.642891,11.306277,-5.342796,...,0.048174,0.432347,0.107536,0.282098,-0.489400,-1.177175,0.571867,0.756571,0.298008,-0.371687
3,triviaqa_00003_t0.1_ans00,truthfulqa,-38.254707,16.724350,5.560056,4.334621,-2.047990,-1.049951,-4.873804,0.018275,...,-0.599921,0.486538,-0.425430,-0.350028,0.252028,0.711712,0.052531,0.146374,-0.299138,0.118096
4,triviaqa_00004_t0.1_ans00,truthfulqa,-45.817905,12.770360,-13.607718,-6.582739,-5.345684,8.146999,-3.381339,-1.895386,...,-0.064618,-0.038639,-0.014426,0.069018,0.539320,0.281209,0.443333,-0.029841,0.102833,0.510708


In [9]:
import torch

torch.save(
    {
        "X": torch.tensor(
            df_pca.filter(like="PC").values,
            dtype=torch.float32
        ),
        "id": df_pca["id"].tolist(),
        "dataset": df_pca["dataset"].tolist()
    },
    "../data/processed/df_pca.pt"
)